# Notebook 3: Neural Models (CL-BiLSTM and WC-Hybrid)
**Dissertation:** Morphological Modeling of MSA and DA Using NLP with Deep Learning  
**Author:** Mohamed Medouani | Capitol Technology University

This notebook implements the two neural network architectures:
- **CL-BiLSTM:** Character-level BiLSTM (PyTorch 1.9.0) — 128d embeddings, 512 hidden units, 2 layers
- **WC-Hybrid:** Word-Character Hybrid (TensorFlow 2.6.0) — 300d word + 128d char, CNN + BiLSTM

## Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

## 1. Character-Level BiLSTM (CL-BiLSTM)
**Framework:** PyTorch 1.9.0  
**Architecture:** Char embedding (128d) → 2-layer BiLSTM (512 hidden) → Linear classifier  
**Training:** 30 epochs, batch size 64, Adam optimizer  
**MSA WLA:** 88.7% (Table 12)

In [ ]:
class CL_BiLSTM(nn.Module):
    """
    Character-Level Bidirectional LSTM for Arabic morphological tagging.
    Processes input text character by character to capture sub-word morphological patterns.
    Particularly effective for Arabic due to its root-and-pattern morphology (PALM).
    
    Architecture (from Chapter 4):
    - Embedding: 128-dimensional character embeddings
    - BiLSTM: 2 layers, 512 hidden units per direction
    - Output: Linear layer mapping to morphological tag space
    """
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=512, n_layers=2, num_tags=15, dropout=0.3):
        super(CL_BiLSTM, self).__init__()
        self.embedding_dim = embedding_dim
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers
        
        # Character embedding layer
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        # Bidirectional LSTM
        self.bilstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            bidirectional=True,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0
        )
        
        # Dropout for regularization (rate: 0.1-0.3 per Chapter 4)
        self.dropout = nn.Dropout(dropout)
        
        # Output classification layer
        self.fc = nn.Linear(hidden_dim * 2, num_tags)  # *2 for bidirectional

    def forward(self, x):
        # x: (batch_size, seq_len)
        embedded = self.dropout(self.embedding(x))
        # embedded: (batch_size, seq_len, embedding_dim)
        
        lstm_out, (hidden, cell) = self.bilstm(embedded)
        # lstm_out: (batch_size, seq_len, hidden_dim * 2)
        
        lstm_out = self.dropout(lstm_out)
        logits = self.fc(lstm_out)
        # logits: (batch_size, seq_len, num_tags)
        return logits

# --- Model Initialization ---
VOCAB_SIZE = 30000  # Arabic character vocabulary
EMBEDDING_DIM = 128  # From dissertation
HIDDEN_DIM = 512     # From dissertation
N_LAYERS = 2         # From dissertation
NUM_TAGS = 15
DROPOUT = 0.3        # From dissertation (range 0.1-0.3)

model = CL_BiLSTM(VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, N_LAYERS, NUM_TAGS, DROPOUT).to(device)
print("CL-BiLSTM Architecture:")
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

In [ ]:
def train_cl_bilstm(model, train_loader, epochs=30, lr=1e-3):
    """
    Training loop for CL-BiLSTM.
    Hyperparameters from Chapter 4:
    - Epochs: 30
    - Batch size: 64
    - Optimizer: Adam
    - Gradient clipping: max_norm=1.0
    - Label smoothing: 0.1
    """
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss(ignore_index=-100, label_smoothing=0.1)
    
    # Learning rate scheduler: linear warmup + decay
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr, steps_per_epoch=len(train_loader), epochs=epochs
    )
    
    train_losses = []
    
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        
        for batch in train_loader:
            input_ids = batch[0].to(device)
            labels = batch[1].to(device)
            
            optimizer.zero_grad()
            logits = model(input_ids)
            
            # Reshape for loss: (batch * seq_len, num_tags) vs (batch * seq_len)
            loss = criterion(logits.view(-1, NUM_TAGS), labels.view(-1))
            
            # Gradient clipping (max_norm=1.0 from Chapter 4)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            epoch_loss += loss.item()
        
        avg_loss = epoch_loss / len(train_loader)
        train_losses.append(avg_loss)
        
        if (epoch + 1) % 5 == 0:
            print(f"Epoch [{epoch+1}/{epochs}] | Loss: {avg_loss:.4f} | LR: {scheduler.get_last_lr()[0]:.6f}")
    
    return train_losses

# --- Demonstration with dummy data ---
BATCH_SIZE = 64
SEQ_LEN = 50

dummy_inputs = torch.randint(1, VOCAB_SIZE, (BATCH_SIZE * 2, SEQ_LEN))
dummy_labels = torch.randint(0, NUM_TAGS, (BATCH_SIZE * 2, SEQ_LEN))
dummy_dataset = TensorDataset(dummy_inputs, dummy_labels)
dummy_loader = DataLoader(dummy_dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Running 3 training epochs as demonstration...")
losses = train_cl_bilstm(model, dummy_loader, epochs=3)
print(f"Final demo loss: {losses[-1]:.4f}")

## 2. Word-Character Hybrid (WC-Hybrid) Model
**Framework:** TensorFlow 2.6.0 / Keras  
**Architecture:** Word embedding (300d) + CNN char features (128d) → BiLSTM (512) → Hierarchical classifier  
**Training:** 25 epochs, batch size 32, AdamW optimizer  
**MSA WLA:** 90.2% (Table 12)

In [ ]:
class WC_Hybrid(nn.Module):
    """
    Word-Character Hybrid model for Arabic morphological tagging.
    Combines word-level semantic representations with character-level morphological features.
    
    Architecture (from Chapter 4):
    - Word embeddings: 300-dimensional
    - Character embeddings: 128-dimensional
    - Character CNN: multiple filter sizes to capture character n-gram patterns
    - Word BiLSTM: 512 hidden units
    - Concatenation + hierarchical classification
    """
    def __init__(self, word_vocab_size, char_vocab_size,
                 word_emb_dim=300, char_emb_dim=128,
                 cnn_filters=256, cnn_kernel_sizes=(2, 3, 4),
                 bilstm_hidden=512, num_tags=15, dropout=0.3):
        super(WC_Hybrid, self).__init__()
        
        # Word-level processing
        self.word_embedding = nn.Embedding(word_vocab_size, word_emb_dim, padding_idx=0)
        self.word_bilstm = nn.LSTM(
            word_emb_dim, bilstm_hidden,
            bidirectional=True, batch_first=True
        )
        
        # Character-level processing: CNN with multiple kernel sizes
        self.char_embedding = nn.Embedding(char_vocab_size, char_emb_dim, padding_idx=0)
        self.char_convs = nn.ModuleList([
            nn.Conv1d(char_emb_dim, cnn_filters // len(cnn_kernel_sizes), k)
            for k in cnn_kernel_sizes
        ])
        
        self.dropout = nn.Dropout(dropout)
        
        # Combined output dimension: bilstm (2*hidden) + CNN features (cnn_filters)
        combined_dim = bilstm_hidden * 2 + cnn_filters
        
        # Hierarchical classification layers
        self.fc1 = nn.Linear(combined_dim, combined_dim // 2)
        self.fc2 = nn.Linear(combined_dim // 2, num_tags)
        self.relu = nn.ReLU()

    def forward(self, word_input, char_input):
        # Word processing
        word_emb = self.dropout(self.word_embedding(word_input))
        word_out, _ = self.word_bilstm(word_emb)
        # word_out: (batch, seq_len, bilstm_hidden * 2)
        
        # Character processing
        char_emb = self.char_embedding(char_input)
        # char_input: (batch, seq_len, char_seq_len) -> process each word's chars
        batch, seq_len, char_seq = char_emb.shape[0], char_emb.shape[1], char_emb.shape[2]
        char_emb_flat = char_emb.view(batch * seq_len, char_seq, -1).permute(0, 2, 1)
        
        # Apply CNN filters
        char_features = []
        for conv in self.char_convs:
            if char_emb_flat.shape[2] >= conv.kernel_size[0]:
                c = torch.relu(conv(char_emb_flat))
                c = torch.max(c, dim=2)[0]  # Global max pooling
                char_features.append(c)
        
        char_out = torch.cat(char_features, dim=1).view(batch, seq_len, -1)
        # char_out: (batch, seq_len, cnn_filters)
        
        # Concatenate word and char representations
        combined = torch.cat([word_out, char_out], dim=2)
        combined = self.dropout(combined)
        
        # Hierarchical classification
        out = self.relu(self.fc1(combined))
        out = self.dropout(out)
        logits = self.fc2(out)
        return logits

# --- Model Initialization ---
WORD_VOCAB = 50000
CHAR_VOCAB = 1000
wc_model = WC_Hybrid(WORD_VOCAB, CHAR_VOCAB).to(device)
print("WC-Hybrid Architecture:")
print(wc_model)
total_params = sum(p.numel() for p in wc_model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

## Performance Summary: Neural Models vs Baselines

In [ ]:
import pandas as pd

results = {
    'Model': ['RB (Rule-Based)', 'SB (Statistical)', 'CL-BiLSTM', 'WC-Hybrid'],
    'MSA WLA (%)': [82.1, 85.4, 88.7, 90.2],
    'Framework': ['Custom Python', 'Custom Python', 'PyTorch 1.9.0', 'TensorFlow 2.6.0'],
    'Epochs': ['N/A', 'N/A', '30', '25'],
    'Batch Size': ['N/A', 'N/A', '64', '32'],
    'Optimizer': ['N/A', 'N/A', 'Adam', 'AdamW'],
    'Params': ['N/A', 'N/A', '~15M', '~22M']
}
df = pd.DataFrame(results)
print(df.to_string(index=False))